# Human review — reviewer 1

**This notebook is for `reviewer_1` only.** Your reviewer ID is already configured;
you do not need to select it. Your answers are saved only to `reviewer_1.csv` in the
frozen packet folder. The other reviewer uses their own notebook and answer file.

You will review the same 100 articles as the other reviewer, separately for both models:
**200 article–model items**. An unassigned BERTopic item receives qualitative flags but
no numeric fit rating. Work independently and do not inspect the other reviewer's answers.

### Your steps

1. Confirm that the coordinator has prepared the frozen packet and your answer CSV.
2. Run Step 1 to load your own file. The default packet location follows the selected run;
   change `PACKET_FOLDER` only if the coordinator gave you another folder.
3. Read the rubric, then display the next unfinished article–topic item.
4. Read the full article, enter your own answers, and explicitly save that item.
5. Repeat the display and answer-entry cells; use the progress cell to pause or finish.
6. Give your CSV to the coordinator when complete. Combined results and interpretation
   are handled in [review_coordinator.ipynb](review_coordinator.ipynb), after both reviewers finish.

Run cells with `Shift+Enter`. After setup, edit only the item selector (if correcting an
old answer) and the commented answer-entry fields. Save switches start off, and answers
are never filled in automatically. Do not use Run All as a substitute for rating each item.

## Before you start

The coordinator must finish model selection and topic naming for both methods, then run
notebook 03 to create the frozen joint-review packet. If these inputs are missing, the
notebook reports **pending** and does not create replacement packets or ratings.

Your packet folder needs these files:

- `review_packet.csv` — the common article text and frozen model assignments.
- `review_freeze.json` and `review_status.json` — packet identity and readiness checks.
- `reviewer_1.csv` — your own editable answer file.

You do **not** need the other reviewer's CSV to complete your work. The coordinator keeps
both independent files for the final comparison. Models, labels, article IDs, and the sample
must not be changed during review.

Use `uv run jupyter lab` from the repository environment. Before entering real ratings,
save a personal working copy as `reviewer_1_workbook.ipynb` inside the packet's `human_review`
folder. These working copies are ignored by Git; keep this checked-in notebook as a blank
template. Your copy may contain answers in edited code cells as well as article text in outputs.

The notebooks are in `final-project/code/human-review/`. By default, the packet and answer
CSVs remain under `final-project/code/modern-methods/Results/comparison/<selected-run>/human_review/`.
This split does not move or overwrite saved answers. The coordinator can use
[review_coordinator.ipynb](review_coordinator.ipynb) for the full preparation and summary instructions.

In [1]:
from pathlib import Path
import html
import json
import sys
import pandas as pd
from IPython.display import HTML, display

# Locate the project whether Jupyter opened from the repository or notebook folder.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ROOT = REPO / "final-project/code/modern-methods"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from src.reviewer import load_session, choose_item, save_answer


## Step 1 — load reviewer 1's file

Your ID is fixed below as `"reviewer_1"`. **Leave it unchanged.** The other person should
open their own reviewer notebook instead of changing this ID.

Normally leave `PACKET_FOLDER = None`; the notebook finds the currently selected run.
If the coordinator supplied a separate local packet folder, enter its absolute path instead.

This step validates the frozen labels and your existing answers. It reads **only your answer
file**. Missing inputs produce a pending message; altered packet identities or invalid answers
raise an error for the coordinator to investigate.

In [2]:
# Preconfigured for this notebook. Leave this reviewer ID unchanged.
REVIEWER = "reviewer_1"

# OPTIONAL: only change this if the coordinator gave you a separate packet folder.
PACKET_FOLDER = None              # Example: Path("/absolute/path/to/human_review")

SESSION = None
CURRENT_ITEM = None
REVIEW_FOLDER = None

if PACKET_FOLDER is not None:
    REVIEW_FOLDER = Path(PACKET_FOLDER).expanduser().resolve()
else:
    selection_file = ROOT / "Results/modern/selection.json"
    selection = json.loads(selection_file.read_text()) if selection_file.exists() else {}
    if selection.get("run_id"):
        REVIEW_FOLDER = ROOT / "Results/comparison" / selection["run_id"] / "human_review"

if REVIEW_FOLDER is None:
    print("Pending: the coordinator must finish model selection, labels, and notebook 03's review-packet step.")
else:
    required = ["review_packet.csv", "review_freeze.json", "review_status.json", f"{REVIEWER}.csv"]
    missing = [name for name in required if not (REVIEW_FOLDER / name).exists()]
    if missing:
        print("Pending review files:", ", ".join(missing))
        print("Ask the coordinator to supply the frozen packet and your own answer file.")
    else:
        SESSION = load_session(REVIEW_FOLDER, REVIEWER)
        print(f"Opened {REVIEWER}: {int(SESSION['complete'].sum())}/{len(SESSION['packet'])} items complete.")
        print("Your answers are saved in:", SESSION["path"])

Pending: the coordinator must finish model selection, labels, and notebook 03's review-packet step.


## Step 2 — use the same rubric throughout

Judge **how well the assigned label describes the article's content**, not whether you agree
with the article or whether its author supports autonomous vehicles. Read the whole article;
do not score from the title or a few matching words alone.

| Fit rating | Meaning | How to decide |
| --- | --- | --- |
| **1** | Unrelated or misleading | The label does not describe the article's subject. |
| **2** | Weak or incidental fit | The label fits a passing mention or a minor part of the article. |
| **3** | Partial or mixed fit | The label describes a meaningful portion, but misses substantial content. |
| **4** | Good fit | The label describes a main theme with minor omissions or imprecision. |
| **5** | Clear central-theme fit | The label accurately and specifically describes a central theme. |

Enter these fields for each item:

| Field | What to enter |
| --- | --- |
| `FIT_RATING` | A whole number 1–5; use `None` for topic −1 (unassigned). |
| `IRRELEVANT` | `True` if the article is not substantively about autonomous/self-driving vehicles; otherwise `False`. An incidental passing mention can be irrelevant. This flag concerns the article, not the model. |
| `AMBIGUOUS_LABEL` | `True` if the wording is unclear, overly broad, or could reasonably mean different things; otherwise `False`. A clearly worded but incorrect label is a poor fit, not automatically ambiguous. For an unassigned item, use `False` because there is no topic label to assess. |
| `NOTES` | Your brief reason or an important limitation. Notes are encouraged for low scores, ambiguity, irrelevance, and unassigned items. Use article IDs and paraphrases rather than copying long passages. |

An article may be irrelevant to autonomous driving yet fit its assigned topic; keep those
judgments separate. Rate each model's label on its own merits, not against the other model.
Do not force the same score for the two models.

**Unassigned example (not a study result):** if the displayed topic is −1, leave
`FIT_RATING = None`, set the two flags, and note what the article discusses. Do not give it a
rating of 1 merely because the model did not assign a topic.

Work independently. Do not use generated scores, model metrics, or another reviewer's ratings
to decide your own. The packets identify the models, so this evaluation is not blinded. If you
recognize an article from earlier development work, record that in your notes and tell the
coordinator; do not silently replace the article or edit the frozen labels.

## Step 3 — display one item and read the full article

Leave `ITEM_NUMBER = None` to open the next unfinished item. To inspect or correct a previous
answer, enter its **1-based item number**. The display includes your existing answers, if any.
Scroll through the article box to read the full text; it is not shortened.

After saving an answer in Step 4, rerun **this cell** to move to the next item.

In [3]:
# Normally leave this as None. For a correction, enter the displayed item number.
ITEM_NUMBER = None
CURRENT_ITEM = None

if SESSION is None:
    print("Finish Step 1 before displaying an article.")
else:
    # Reload your own CSV so an interrupted session resumes from saved answers.
    SESSION = load_session(SESSION["folder"], SESSION["reviewer"])
    CURRENT_ITEM = choose_item(SESSION, ITEM_NUMBER)
    if CURRENT_ITEM is None:
        print("All of your items are complete. Continue to Step 5.")
    else:
        article = SESSION["packet"].iloc[CURRENT_ITEM]
        print(f"ITEM {CURRENT_ITEM + 1} of {len(SESSION['packet'])} — copy this item number into ITEM_TO_SAVE below.")
        display(pd.Series({"article_id": article.article_id, "model": article.model,
                           "topic_id": article.topic_id, "assigned_label": article.label,
                           "title": article.title}, name="Review this assignment"))
        if article.topic_id == -1:
            print("UNASSIGNED: leave FIT_RATING = None; complete the qualitative flags and notes.")
        # Escape article text before rendering: article HTML must be displayed as text.
        display(HTML('<div style="white-space:pre-wrap;max-height:650px;overflow:auto;'
                     'padding:18px;border:1px solid #aaa;line-height:1.6">'
                     + html.escape(str(article.text)) + '</div>'))
        display(SESSION["answers"].loc[CURRENT_ITEM, ["fit_rating", "irrelevant", "ambiguous_label", "notes"]].rename("Your saved answers"))

Finish Step 1 before displaying an article.


## Step 4 — fill in your answer and save it

Edit the values below for the item you just read. Python uses `True`, `False`, and `None`
with capital letters; put written notes inside quotes. Copy the displayed item number into
`ITEM_TO_SAVE`, then set `SAVE_RATING = True` and run the cell.

The item-number check prevents an old answer-entry cell from saving against a newly displayed
article. Only your own row is updated. Saving fails if the packet changed, another session
changed your CSV, a rating is invalid, or you attempt to overwrite an answer without opting in.
For an intentional correction only, set `ALLOW_UPDATE = True` after displaying that item.

**Loop:** display the next item → read → edit every answer field and the item number → save.
Simply running every cell is not a substitute for completing the reviews.

In [4]:
# FILL THESE IN for the displayed item; no answers are supplied automatically.
ITEM_TO_SAVE = None          # Copy the displayed ITEM number here (e.g., 1).
FIT_RATING = None            # 1, 2, 3, 4, or 5. Keep None only for an unassigned item.
IRRELEVANT = None            # Replace with True or False.
AMBIGUOUS_LABEL = None       # Replace with True or False.
NOTES = ""                  # Write your reason or any uncertainty in your own words.

SAVE_RATING = False          # Change to True when this item's answers are ready to save.
ALLOW_UPDATE = False         # True only when intentionally correcting existing answers.

if not SAVE_RATING:
    print("Nothing saved. Read the item and fill in the fields before enabling SAVE_RATING.")
elif SESSION is None or CURRENT_ITEM is None:
    print("Nothing saved. Display an item in Step 3 first.")
elif type(ITEM_TO_SAVE) is not int or ITEM_TO_SAVE != CURRENT_ITEM + 1:
    raise ValueError("ITEM_TO_SAVE must match the displayed item's number. Check it before saving.")
else:
    SESSION = save_answer(SESSION, CURRENT_ITEM, FIT_RATING, IRRELEVANT, AMBIGUOUS_LABEL,
                          notes=NOTES, allow_update=ALLOW_UPDATE)
    print(f"Saved item {ITEM_TO_SAVE} to {SESSION['reviewer']}.csv.")
    print(f"Progress: {int(SESSION['complete'].sum())}/{len(SESSION['packet'])} items complete.")
    print("Rerun Step 3 to display the next item, then update all answer fields.")
    CURRENT_ITEM = None      # A new display is required before another save.
SAVE_RATING = False

Nothing saved. Read the item and fill in the fields before enabling SAVE_RATING.


## Step 5 — check progress, take a break, or finish

This cell validates **your own** file and lists unfinished item numbers. Blank flags remain
unfinished, even for unassigned articles. Notes are optional, but useful explanations should
accompany difficult judgments.

To take a break, save your current item, then close Jupyter. To resume, reopen the notebook,
run Steps 1–3 in this reviewer notebook, and leave `ITEM_NUMBER = None`.

When complete, give the coordinator your `reviewer_1.csv` through the team's
usual file-sharing process. Keep the original independent ratings; discuss disagreements only
after both reviewers finish. Do not overwrite them with consensus ratings.

Your personal workbook may contain full article text in outputs and ratings in edited code.
Share your CSV with the coordinator, not your working notebook. If preparing a blank notebook
template to share, reset the answer/configuration cells and save switches **as well as** using
**Kernel → Restart Kernel and Clear Outputs** (wording varies by Jupyter version). Clearing
outputs alone does not erase ratings typed into code cells.

In [5]:
if SESSION is None:
    print("Load your assigned review file in Step 1 first.")
else:
    SESSION = load_session(SESSION["folder"], SESSION["reviewer"])
    total = len(SESSION["packet"])
    finished = int(SESSION["complete"].sum())
    print(f"{SESSION['reviewer']}: {finished}/{total} items complete ({finished / total:.0%}).")
    remaining = SESSION["answers"].loc[~SESSION["complete"], ["article_id", "model", "topic_id"]].copy()
    remaining.insert(0, "item_number", remaining.index + 1)
    if remaining.empty:
        print("Your file is complete and valid. Tell the coordinator; keep your independent ratings unchanged.")
    else:
        print("Next unfinished items (first 20):")
        display(remaining.head(20))
    print("Your saved file:", SESSION["path"])

Load your assigned review file in Step 1 first.
